# Cấp độ 1 — Knowledge Tracing trên đồ thị (Kaggle GPU)

Notebook **clone repo** [Knowledge-Tracing](https://github.com/linhnnh688/Knowledge-Tracing) rồi gọi hàm từ package `level1/` để triển khai thực nghiệm tuần tự:

| Yêu cầu đề bài (Cấp độ 1) | Triển khai thực nghiệm tuần tự |
|---|---|
| **1. Chuẩn bị dữ liệu & chống rò rỉ** | Chia tập `train/val/test` theo user (cố định seed=42), Transition graph chỉ xây từ tập train |
| **2. Baseline không dùng đồ thị** | **DKT** (Deep Knowledge Tracing với LSTM) |
| **3. Mô hình dùng đồ thị** | **GKT** (Graph-based Knowledge Tracing với Transition Graph) |
| **4. Ablation cấu trúc đồ thị** | **GKT-Identity** (ma trận đơn vị) và **GKT-Zero** (ma trận 0) |
| **5. Mở rộng (tùy chọn)** | **GIKT** (Bipartite question-skill graph) & **GIKT-empty** |
| **6. So sánh & trực quan hóa** | Bảng metrics (AUC, Acc, Loss), biểu đồ quá trình học & biểu đồ cột |
| **7. Giải thích cơ bản (xAI)** | Phân tích 5 case đúng + 5 case sai kèm lịch sử tương tác & láng giềng trên đồ thị |

Bản gốc `GIKT/` là TensorFlow 1.12 (`tf.contrib`) — không tương thích Kaggle hiện nay. Logic Cấp 1 đã được chuẩn hóa trong `level1/`, notebook đóng vai trò điều phối tuần tự.

## 0. Clone repo và cấu hình môi trường

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/linhnnh688/Knowledge-Tracing.git'
KAGGLE_ROOT = Path('/kaggle/working')
IS_KAGGLE = KAGGLE_ROOT.exists()

if IS_KAGGLE:
    REPO = KAGGLE_ROOT / 'Knowledge-Tracing'
    if not (REPO / '.git').exists():
        if REPO.exists():
            shutil.rmtree(REPO, ignore_errors=True)
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)])
    else:
        try:
            subprocess.check_call(['git', '-C', str(REPO), 'fetch', 'origin', 'main', '--depth', '1'])
            subprocess.check_call(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'])
        except Exception as e:
            print(f'Git sync error ({e}), re-cloning...')
            shutil.rmtree(REPO, ignore_errors=True)
            subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)])
else:
    REPO = Path.cwd() if (Path.cwd() / 'level1').exists() else Path.cwd() / 'Knowledge-Tracing'

os.chdir(REPO)
GKT_DIR = REPO / 'GKT'

# Đảm bảo REPO và REPO/GKT luôn nằm trong sys.path để tránh lỗi import module
for p in [REPO, GKT_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print('Repo directory:', REPO.resolve())
print('GKT directory:', GKT_DIR.resolve())


In [ ]:
# Kaggle GPU đã có sẵn PyTorch GPU — không pip install torch
%pip install -q pandas scipy scikit-learn matplotlib tqdm

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from level1.pipeline import load_kt_data, save_split, run_experiment, run_gikt_experiment
from level1.explain import collect_cases, format_cases, skill_lookup_from_loader
from level1.dashboard import export_learner_view, interactive_learner_view, render_learner_view

print("PyTorch:", torch.__version__, "| CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

# Cấu hình tham số thực nghiệm:
# Đặt QUICK_RUN = True để chạy nhanh kiểm thử (assistment_test15.csv — CHỈ dùng để smoke test, KHÔNG dùng cho báo cáo)
# Đặt QUICK_RUN = False để chạy toàn bộ dataset ASSISTments 2009 (skill_builder_data.csv) — BẮT BUỘC cho kết quả báo cáo
QUICK_RUN = False
DATA_FILE = "assistment_test15.csv" if QUICK_RUN else "skill_builder_data.csv"
EPOCHS = 8 if QUICK_RUN else 20
BATCH_SIZE = 16 if QUICK_RUN else 32
MAX_SEQ_LEN = 80 if QUICK_RUN else 200
SAVE_DIR = Path("/kaggle/working/level1_out") if IS_KAGGLE else (REPO / "outputs" / "level1")

print(f"Config: data={DATA_FILE}, epochs={EPOCHS}, batch_size={BATCH_SIZE}, max_seq_len={MAX_SEQ_LEN}")
print("Output dir:", SAVE_DIR)


## 1. Tổng quan cơ chế: Baseline DKT vs Graph-based GKT

* **DKT (Deep Knowledge Tracing - Piech et al., NeurIPS 2015):**
  * Mô hình tuần tự truyền thống không dùng đồ thị.
  * Nhận chuỗi cặp `(khái niệm, kết quả trả lời)` qua LSTM/GRU để cập nhật trạng thái ẩn $h_t$, sau đó chiếu ra xác suất làm đúng khái niệm tiếp theo $P(r_{t} = 1 \mid c_t)$.
  * Điểm yếu: Không mô hình hóa mối quan hệ liên kết giữa các khái niệm kiến thức (prerequisite, co-occurrence, transition).

* **GKT (Graph-based Knowledge Tracing - Nakagawa et al., WI 2019):**
  * Mô hình theo dõi tri thức dựa trên đồ thị: mỗi đỉnh là một khái niệm kiến thức (KC) có trạng thái ẩn riêng $h_c^t$.
  * Tại mỗi bước tương tác với KC $c$, GKT dùng mạng nơ-ron đồ thị (GNN) lan truyền thông tin sang các KC láng giềng trên ma trận kề $A$ (ví dụ: `Transition Graph`).
  * Sử dụng Erase-Add Gate và GRUCell để cập nhật trạng thái của tất cả các KC, từ đó dự đoán xác suất trả lời đúng.

## 2. Khám phá dữ liệu (EDA)

In [ ]:
df = pd.read_csv(REPO / "GKT" / "data" / DATA_FILE, low_memory=False)
print(f"Shape: {df.shape} | Users: {df['user_id'].nunique()} | Skills: {df['skill_id'].nunique()}")
print(f"Tỷ lệ trả lời đúng trung bình: {df['correct'].eq(1).mean():.3%}")

user_len = df.groupby("user_id").size()
fig, ax = plt.subplots(figsize=(8, 3))
user_len.hist(bins=40, ax=ax, edgecolor="black", alpha=0.7)
ax.set_title(f"Phân phối số tương tác mỗi người học ({DATA_FILE})")
ax.set_xlabel("Số lượt tương tác")
ax.set_ylabel("Số lượng học sinh")
plt.tight_layout()
plt.show()


## 3. Triển khai thực nghiệm tuần tự (Sequential Experiment Pipeline)

Quy trình tuần tự gồm các bước chuẩn mực:
1. **Bước 3.1:** Nạp dữ liệu và tạo split cố định theo user (`split.json`), sinh đồ thị Transition chỉ từ tập train để chống rò rỉ dữ liệu.
2. **Bước 3.2:** Huấn luyện **Baseline DKT** (chuẩn đối sánh không đồ thị).
3. **Bước 3.3:** Huấn luyện **GKT với Transition Graph** (mô hình đồ thị chính).
4. **Bước 3.4:** Huấn luyện **GKT Ablations** (`Identity` và `Zero Graph`) để kiểm chứng cấu trúc đồ thị.
5. **Bước 3.5 (Tùy chọn):** Huấn luyện **GIKT** (Bipartite Question-Skill Graph) mở rộng.

### 3.1. Chuẩn bị dữ liệu và chia tập cố định (User-wise Split & Anti-Leakage)

In [ ]:
# Danh sách lưu kết quả của tất cả các mô hình
results = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Nạp dữ liệu với đồ thị Transition (tính toán strictly từ tập train)
print(">>> [Bước 3.1] Chuẩn bị dữ liệu và chia tập cố định...")
shared = load_kt_data(
    DATA_FILE,
    batch_size=BATCH_SIZE,
    graph_type="Transition",
    model_type="GKT",
    seed=42,
    max_seq_len=MAX_SEQ_LEN,
    use_cuda=torch.cuda.is_available(),
)

# Lưu tập chia train/val/test để đảm bảo tính tái lập và minh chứng chống rò rỉ dữ liệu
split = save_split(shared, SAVE_DIR / "split.json")
print(f"Tổng số khái niệm kiến thức (Concepts): {shared['concept_num']}")
print(f"User split: Train={len(split['train_users'])} | Val={len(split['val_users'])} | Test={len(split['test_users'])}")
print(f"Đã lưu thông tin split tại: {SAVE_DIR / 'split.json'}")

### 3.2. Huấn luyện Baseline không đồ thị: DKT (Deep Knowledge Tracing)

In [ ]:
print(">>> [Bước 3.2] Huấn luyện Baseline DKT (Không đồ thị)...")
res_dkt = run_experiment(
    DATA_FILE,
    model_type="DKT",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=1e-3,
    hid_dim=32,
    emb_dim=32,
    seed=42,
    max_seq_len=MAX_SEQ_LEN,
    save_dir=SAVE_DIR,
    bundle=shared,
    device=device,
)
results.append(res_dkt)
print(f"==> Kết quả Baseline DKT: Test AUC = {res_dkt['test_auc']:.4f} | Test Acc = {res_dkt['test_acc']:.4f} | Loss = {res_dkt['test_loss']:.4f}")

### 3.3. Huấn luyện Mô hình đồ thị: GKT (Graph-based Knowledge Tracing với Transition Graph)

In [ ]:
print(">>> [Bước 3.3] Huấn luyện GKT với Đồ thị Chuyển tiếp (Transition Graph)...")
res_gkt_trans = run_experiment(
    DATA_FILE,
    model_type="GKT",
    graph_type="Transition",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=1e-3,
    hid_dim=32,
    emb_dim=32,
    seed=42,
    max_seq_len=MAX_SEQ_LEN,
    save_dir=SAVE_DIR,
    bundle=shared,
    device=device,
)
results.append(res_gkt_trans)
print(f"==> Kết quả GKT-Transition: Test AUC = {res_gkt_trans['test_auc']:.4f} | Test Acc = {res_gkt_trans['test_acc']:.4f} | Loss = {res_gkt_trans['test_loss']:.4f}")

diff_auc = res_gkt_trans["test_auc"] - res_dkt["test_auc"]
diff_acc = res_gkt_trans["test_acc"] - res_dkt["test_acc"]
print(f"-> Chênh lệch so với Baseline DKT: AUC {diff_auc:+.4f} | Accuracy {diff_acc:+.4f}")

### 3.4. Khảo sát cấu trúc đồ thị & Ablation (Dense, Identity & Zero Graph)

Theo yêu cầu đề bài: *Khảo sát vai trò của cấu trúc liên kết tri thức thông qua các biến thể đồ thị:*
- **Dense Graph**: Đồ thị liên thông toàn bộ với trọng số đồng nhất (Fully-connected baseline).
- **Identity Graph (Ablation)**: Ma trận đơn vị (không có lan truyền giữa các khái niệm khác nhau, mỗi KC chỉ cập nhật trạng thái tự thân).
- **Zero Graph (Ablation)**: Ma trận 0 (ngắt hoàn toàn lan truyền đồ thị).
- **Shuffled Graph (Ablation)**: Giữ nguyên độ thưa (sparsity) của Transition Graph nhưng hoán vị ngẫu nhiên các cạnh — phá vỡ cấu trúc có ý nghĩa, giữ nguyên số lượng cạnh.


In [ ]:
print(">>> [Bước 3.4] Thực hiện Khảo sát đồ thị & Graph Ablation trên GKT (Dense, Identity, Zero, Shuffled)...")

for graph_type in ["Dense", "Identity", "Zero", "Shuffled"]:
    print(f"\n--- Đang huấn luyện GKT với đồ thị {graph_type} ---")
    bundle_abl = load_kt_data(
        DATA_FILE,
        batch_size=BATCH_SIZE,
        graph_type=graph_type,
        model_type="GKT",
        seed=42,
        max_seq_len=MAX_SEQ_LEN,
        use_cuda=device.type == "cuda",
    )
    res_abl = run_experiment(
        DATA_FILE,
        model_type="GKT",
        graph_type=graph_type,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=1e-3,
        hid_dim=32,
        emb_dim=32,
        seed=42,
        max_seq_len=MAX_SEQ_LEN,
        save_dir=SAVE_DIR,
        bundle=bundle_abl,
        device=device,
    )
    results.append(res_abl)
    print(f"==> Kết quả GKT-{graph_type}: Test AUC = {res_abl['test_auc']:.4f} | Test Acc = {res_abl['test_acc']:.4f}")


### 3.5. (Tùy chọn) Mở rộng mô hình đồ thị câu hỏi – kỹ năng: GIKT & GIKT-empty

GIKT xây dựng đồ thị lưỡng phân câu hỏi - kỹ năng (bản PyTorch chuẩn hóa).

In [ ]:
# Cấp độ 1 tập trung chuẩn hóa so sánh Baseline DKT vs GKT (kèm các biến thể đồ thị).
# GIKT là mô hình mở rộng đồ thị lưỡng phân câu hỏi - kỹ năng (tốn nhiều RAM/thời gian khi chạy full dataset).
# Mặc định đặt False để tối ưu tốc độ và tài nguyên. Đổi thành True nếu bạn muốn khảo sát thêm GIKT.
RUN_GIKT = False

if RUN_GIKT:
    print(">>> [Bước 3.5] Huấn luyện GIKT (Question-Skill Graph) & GIKT-empty (Ablation)...")
    res_gikt = run_gikt_experiment(
        DATA_FILE,
        split["train_users"],
        split["val_users"],
        split["test_users"],
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=1e-3,
        emb_dim=32,
        hidden_dim=32,
        seed=42,
        max_seq_len=MAX_SEQ_LEN,
        save_dir=SAVE_DIR,
        device=device,
    )
    results.append(res_gikt)

    res_gikt_empty = run_gikt_experiment(
        DATA_FILE,
        split["train_users"],
        split["val_users"],
        split["test_users"],
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=1e-3,
        emb_dim=32,
        hidden_dim=32,
        seed=42,
        max_seq_len=MAX_SEQ_LEN,
        ablation_empty_graph=True,
        save_dir=SAVE_DIR,
        device=device,
    )
    results.append(res_gikt_empty)
else:
    print(">>> [Bước 3.5] Bỏ qua GIKT (RUN_GIKT = False). Tập trung vào so sánh chuẩn DKT vs GKT.")


## 4. Tổng hợp so sánh kết quả & Phân tích vai trò cấu trúc đồ thị

In [ ]:
# 4.1 Bảng tổng hợp số liệu
table = pd.DataFrame([
    {
        "Mô hình": r["name"],
        "Loại đồ thị": r["graph_type"],
        "Test AUC": round(r["test_auc"], 4),
        "Test Accuracy": round(r["test_acc"], 4),
        "Test Loss (NLL)": round(r["test_loss"], 4),
        "Best Val AUC": round(r["best_val_auc"], 4),
    }
    for r in results
])
print("=== BẢNG TỔNG HỢP KẾT QUẢ CẤP ĐỘ 1 ===")
display(table)

# 4.2 Biểu đồ đường loss và AUC qua từng epoch
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for r in results:
    axes[0].plot(r["history"]["val_loss"], label=r["name"])
    axes[1].plot(r["history"]["val_auc"], label=r["name"])

axes[0].set_title("Validation Loss (NLL) theo Epoch", fontsize=12)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Validation AUC theo Epoch", fontsize=12)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("AUC")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4.3 Biểu đồ cột so sánh Test AUC & Accuracy
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(table))
ax.bar([i - 0.2 for i in x], table["Test AUC"], width=0.4, label="Test AUC")
ax.bar([i + 0.2 for i in x], table["Test Accuracy"], width=0.4, label="Test Acc")
ax.set_xticks(list(x))
ax.set_xticklabels(table["Mô hình"], rotation=20)
ax.set_ylim(0, 1.0)
ax.set_title("So sánh hiệu năng giữa Baseline DKT và GKT cùng các biến thể Ablation", fontsize=12)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# 4.4 Phân tích chuyên sâu Ablation đồ thị GKT
print("\n=== PHÂN TÍCH ABLATION VAI TRÒ ĐỒ THỊ TRÊN GKT ===")
gkt_sub = table[table["Mô hình"].str.startswith("GKT")]
display(gkt_sub)

### 4.5. Đánh giá đa seed (Multi-seed evaluation: Mean $\pm$ Std)

Chạy 3 seed ngẫu nhiên (`[42, 123, 456]`) cho **Baseline DKT** và **GKT-Transition** với phân chia dữ liệu độc lập để báo cáo `Mean ± Std` theo đúng chuẩn nghiên cứu khoa học và yêu cầu trong `KE_HOACH.md`.


In [ ]:
# Đặt RUN_MULTI_SEED = True khi cần chạy kiểm chứng 3 seed cho báo cáo chính thức đồ án
RUN_MULTI_SEED = True

if RUN_MULTI_SEED:
    print("=" * 65)
    print(">>> [Bước 4.5] Huấn luyện đa seed (seeds = [42, 123, 456]) cho DKT & GKT (+ablations)...")
    print("=" * 65)
    seeds = [42, 123, 456]
    # CHỐNG RÒ RỈ: nạp bundle MỘT LẦN với split_seed=42 (đã làm ở Bước 3.1), tái sử dụng
    # across all seeds — chỉ set_seed(seed) khác nhau bên trong run_experiment để đổi khởi tạo,
    # KHÔNG load_kt_data(seed=s) vì sẽ làm split+đồ thị đổi theo seed (confound split variance).
    multi_configs = [
        {"model_type": "DKT", "graph_type": "N/A", "bundle": shared},
        {"model_type": "GKT", "graph_type": "Transition", "bundle": shared},
    ]
    # Nạp thêm bundle cho các ablation graph với CÙNG split_seed=42 (không đổi split)
    for abl_gt in ["Identity", "Zero", "Shuffled"]:
        b_abl = load_kt_data(
            DATA_FILE,
            batch_size=BATCH_SIZE,
            graph_type=abl_gt,
            model_type="GKT",
            seed=42,  # fixed split seed — GIỐNG hệt shared, chỉ đồ thị khác
            max_seq_len=MAX_SEQ_LEN,
            use_cuda=device.type == "cuda",
        )
        multi_configs.append({"model_type": "GKT", "graph_type": abl_gt, "bundle": b_abl})
    multi_summary = []

    for cfg in multi_configs:
        m_type = cfg["model_type"]
        g_type = cfg["graph_type"]
        bundle_fixed = cfg["bundle"]  # split đã cố định — KHÔNG load lại theo seed
        name = m_type if g_type == "N/A" else f"GKT-{g_type}"
        aucs, accs, losses = [], [], []

        for s in seeds:
            print(f"--- Đang chạy {name} | seed={s} (split cố định seed=42) ---")
            r = run_experiment(
                DATA_FILE,
                model_type=m_type,
                graph_type=g_type,
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                lr=1e-3,
                hid_dim=32,
                emb_dim=32,
                seed=s,  # chỉ đổi seed khởi tạo weight + thứ tự batch, KHÔNG đổi split
                max_seq_len=MAX_SEQ_LEN,
                save_dir=SAVE_DIR,
                bundle=bundle_fixed,  # tái sử dụng split + đồ thị cố định
                device=device,
            )
            aucs.append(r["test_auc"])
            accs.append(r["test_acc"])
            losses.append(r["test_loss"])

        multi_summary.append({
            "Mô hình": name,
            "Test AUC Mean": float(np.mean(aucs)),
            "Test AUC Std": float(np.std(aucs)),
            "Test Acc Mean": float(np.mean(accs)),
            "Test Acc Std": float(np.std(accs)),
            "Test Loss Mean": float(np.mean(losses)),
            "Test Loss Std": float(np.std(losses)),
        })

    df_multiseed = pd.DataFrame(multi_summary)
    print("\n=== KẾT QUẢ ĐA SEED (MEAN ± STD) — split cố định seed=42 ===")
    for row in multi_summary:
        print(f"{row['Mô hình']:20s} | AUC: {row['Test AUC Mean']:.4f} ± {row['Test AUC Std']:.4f} | Acc: {row['Test Acc Mean']:.4f} ± {row['Test Acc Std']:.4f}")
    display(df_multiseed)
    
    ms_csv = SAVE_DIR / "multiseed_metrics.csv"
    df_multiseed.to_csv(ms_csv, index=False)
    print(f"\n[OK] Đã lưu bảng kết quả đa seed vào: {ms_csv}")
else:
    print(">>> Bỏ qua đánh giá đa seed (RUN_MULTI_SEED = False). Đổi sang True để chạy khi làm báo cáo đồ án.")


## 5. Nhiệm vụ 3 mức cơ bản: Phân tích 5 case đúng + 5 case sai

Trích xuất 5 trường hợp mô hình dự đoán đúng và 5 trường hợp dự đoán sai từ mô hình tốt nhất (`GKT-Transition`), kèm:
- Khái niệm kiến thức mục tiêu cần dự đoán.
- Xác suất dự đoán $\hat{y}$ so với nhãn thực tế $y$.
- Lịch sử làm bài trước đó của người học.
- Các khái niệm liên quan kề cận trên đồ thị chuyển tiếp (Transition Graph).

In [ ]:
# Trích xuất 5 case đúng + 5 case sai từ mô hình GKT-Transition
gkt_best = next(r for r in results if r["name"] == "GKT-Transition")
model = gkt_best["model"].to(device)
bundle = gkt_best["bundle"]
lookup = skill_lookup_from_loader(bundle["test_loader"])
graph = bundle.get("graph")
if graph is not None:
    graph = graph.detach().cpu()

buckets = collect_cases(
    model,
    bundle["test_loader"],
    graph=graph,
    skill_lookup=lookup,
    n_each=5,
    history_k=8,
    device=device,
    model_type="GKT",
)
report = format_cases(buckets)
print(report)

# Lưu báo cáo vào cases.md
case_file = SAVE_DIR / "cases.md"
case_file.write_text(report, encoding="utf-8")
print(f"\n[OK] Đã lưu chi tiết 10 case study vào: {case_file}")


## 5b. Dashboard chọn người học (Nhiệm vụ bổ sung §8)

Chọn một người học trong tập test → xem xác suất dự đoán từng bước, lịch sử làm bài,
trạng thái tri thức ước lượng (chuẩn norm của hidden state $h_t$ theo từng khái niệm) và bằng chứng.

- `export_learner_view` chạy model trên toàn bộ test set, gom theo từng người học, xuất `learner_view.json`.
- `interactive_learner_view` hiển thị dropdown chọn người học nếu có `ipywidgets`; ngược lại in ra người học đầu tiên.


In [ ]:
print(">>> [Bước 5b] Export learner view cho dashboard...")
learner_views = export_learner_view(
    model,
    bundle["test_loader"],
    skill_lookup=lookup,
    graph=graph,
    device=device,
    model_type="GKT",
    out_path=SAVE_DIR / "learner_view.json",
)
print(f"Đã export {len(learner_views)} người học → {SAVE_DIR / 'learner_view.json'}")

# Hiển thị tương tác (dropdown) nếu có ipywidgets, không thì in người học đầu tiên
interactive_learner_view(learner_views)


## 6. Lưu trữ kết quả và xuất Artifacts

## 6. Lưu trữ kết quả và xuất Artifacts

In [ ]:
# Lưu bảng kết quả ra CSV và JSON
out_csv = Path("/kaggle/working/level1_metrics.csv") if IS_KAGGLE else (SAVE_DIR / "metrics.csv")
table.to_csv(out_csv, index=False)
(SAVE_DIR / "metrics.json").write_text(table.to_json(orient="records", indent=2, force_ascii=False), encoding="utf-8")

print("=== DANH SÁCH SẢN PHẨM HOÀN THÀNH ===")
print("1. Metrics CSV:", out_csv)
print("2. Split JSON (chống rò rỉ):", SAVE_DIR / "split.json")
print("3. Case Study xAI Markdown:", SAVE_DIR / "cases.md")
print("4. Model Checkpoints:", list(Path(SAVE_DIR).glob("*.pt")))
if (SAVE_DIR / "multiseed_metrics.csv").exists():
    print("5. Multi-seed Metrics CSV:", SAVE_DIR / "multiseed_metrics.csv")
if (SAVE_DIR / "learner_view.json").exists():
    print("6. Learner View JSON (dashboard):", SAVE_DIR / "learner_view.json")
table
